# Apache Hudi Core Conceptions (3) - Compaction

## 1. Configuration

In [1]:
%%configure -f
{
    "conf" : {
        "spark.jars":"hdfs:///tmp/hudi-spark-bundle.jar",            
        "spark.serializer":"org.apache.spark.serializer.KryoSerializer",
        "spark.sql.extensions":"org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
        "spark.sql.catalog.spark_catalog":"org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
44,application_1678096020253_0064,spark,idle,Link,Link,None,
47,application_1678096020253_0067,spark,idle,Link,Link,None,


In [2]:
%%sh
# deploy hudi bundle jar
hdfs dfs -copyFromLocal -f /usr/lib/hudi/hudi-spark-bundle.jar /tmp/hudi-spark-bundle.jar
hdfs dfs -ls /tmp/hudi-spark-bundle.jar
# deploy hudi-stat.sh - a utility shell script 
wget https://github.com/bluishglc/hudi-core-conceptions/releases/download/v1.0/hudi-stat.sh -O ~/hudi-stat.sh &>/dev/null
chmod a+x ~/hudi-stat.sh
ls ~/hudi-stat.sh

-rw-r--r--   1 emr-notebook hdfsadmingroup   61421977 2023-03-09 10:30 /tmp/hudi-spark-bundle.jar
/home/emr-notebook/hudi-stat.sh


In [3]:
%%html
<style>
table {float:left}
</style>

## 2. Sync Compaction: Inline Schedule + Inline Execute

### 2.1. Create Table

In [4]:
%%sql
set TABLE_NAME=reviews_mor_simple_1

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
49,application_1678096020253_0069,spark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [5]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_1

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [6]:
%env TABLE_NAME=reviews_mor_simple_1

env: TABLE_NAME=reviews_mor_simple_1


In [7]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_1

env: TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_1


In [8]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 3

reviews_mor_simple_1


In [9]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [10]:
%%sql
drop table if exists ${TABLE_NAME}_ro

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [11]:
%%sql
drop table if exists ${TABLE_NAME}_rt

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [12]:
%%sql
create table if not exists ${TABLE_NAME} (
    review_id string, 
    star_rating long, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'mor',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175',
    hoodie.compact.inline = 'true',
    -- hoodie.compact.schedule.inline = 'false',
    hoodie.compact.inline.max.delta.commits = '3'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 2.2. Batch 1 - Insert

In [13]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1999

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [14]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103111861 │ deltacommit │ COMPLETED │ 03-09 10:31 │ 03-09 10:31 │ 03-09 10:31 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪════════

### 2.3.  Batch 2 - Update

In [15]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-01'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [16]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103111861 │ deltacommit │ COMPLETED │ 03-09 10:31 │ 03-09 10:31 │ 03-09 10:31 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309103221933 │ deltacommit │ COMPLETED │ 03-09 10:32 │ 03-09 10:32 │ 03-09 10:32 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        

### 2.4. Batch 3 - Update

In [17]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-02'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [18]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103111861 │ deltacommit │ COMPLETED │ 03-09 10:31 │ 03-09 10:31 │ 03-09 10:31 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309103221933 │ deltacommit │ COMPLETED │ 03-09 10:32 │ 03-09 10:32 │ 03-09 10:32 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309103301774 │ deltacommit │ COMPLETED │ 03-09 10:33 │ 03-09 10:33 │ 03-09 10:33 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309103316928

### 2.5. Batch 4 - Update

In [19]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-03'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [20]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103111861 │ deltacommit │ COMPLETED │ 03-09 10:31 │ 03-09 10:31 │ 03-09 10:31 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309103221933 │ deltacommit │ COMPLETED │ 03-09 10:32 │ 03-09 10:32 │ 03-09 10:32 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309103301774 │ deltacommit │ COMPLETED │ 03-09 10:33 │ 03-09 10:33 │ 03-09 10:33 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309103316928

### 2.6. Batch 5 - Update

In [21]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-04'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [22]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103111861 │ deltacommit │ COMPLETED │ 03-09 10:31 │ 03-09 10:31 │ 03-09 10:31 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309103221933 │ deltacommit │ COMPLETED │ 03-09 10:32 │ 03-09 10:32 │ 03-09 10:32 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309103301774 │ deltacommit │ COMPLETED │ 03-09 10:33 │ 03-09 10:33 │ 03-09 10:33 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309103316928

## 3. Async Compaction: Inline Schedule + Offline Execute

### 3.1. Create Table

In [23]:
%%sql
set TABLE_NAME=reviews_mor_simple_2

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [24]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_2

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [25]:
%env TABLE_NAME=reviews_mor_simple_2

env: TABLE_NAME=reviews_mor_simple_2


In [26]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_2

env: TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_2


In [27]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 3

reviews_mor_simple_2


In [28]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [29]:
%%sql
drop table if exists ${TABLE_NAME}_ro

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [30]:
%%sql
drop table if exists ${TABLE_NAME}_rt

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [31]:
%%sql
create table if not exists ${TABLE_NAME} (
    review_id string, 
    star_rating long, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'mor',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175',
    hoodie.compact.inline = 'false',
    hoodie.compact.schedule.inline = 'true',
    hoodie.compact.inline.max.delta.commits = '3'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 3.2. Batch 1 - Insert

In [32]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1999

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [33]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103534976 │ deltacommit │ COMPLETED │ 03-09 10:35 │ 03-09 10:35 │ 03-09 10:36 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪════════

### 3.3. Batch 2 - Update

In [34]:
%%sql
-- 更新1999-01-01的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-01'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [35]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103534976 │ deltacommit │ COMPLETED │ 03-09 10:35 │ 03-09 10:35 │ 03-09 10:36 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309103631046 │ deltacommit │ COMPLETED │ 03-09 10:36 │ 03-09 10:36 │ 03-09 10:36 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        

### 3.4. Batch 3 - Update

In [36]:
%%sql
-- 更新1999-01-02的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-02'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [37]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103534976 │ deltacommit │ COMPLETED │ 03-09 10:35 │ 03-09 10:35 │ 03-09 10:36 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309103631046 │ deltacommit │ COMPLETED │ 03-09 10:36 │ 03-09 10:36 │ 03-09 10:36 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309103710088 │ deltacommit │ COMPLETED │ 03-09 10:37 │ 03-09 10:37 │ 03-09 10:37 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309103724002

### 3.5. Async Execute Compaction

In [38]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class "org.apache.hudi.utilities.HoodieCompactor" \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'execute' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" > ~/${TABLE_NAME}.compaction.execute.out &>/dev/null

In [39]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103534976 │ deltacommit │ COMPLETED │ 03-09 10:35 │ 03-09 10:35 │ 03-09 10:36 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309103631046 │ deltacommit │ COMPLETED │ 03-09 10:36 │ 03-09 10:36 │ 03-09 10:36 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309103710088 │ deltacommit │ COMPLETED │ 03-09 10:37 │ 03-09 10:37 │ 03-09 10:37 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309103724002

## 4. Async Compaction: Offline Schedule + Offline Execute

### 4.1. Create Table

In [40]:
%%sql
set TABLE_NAME=reviews_mor_simple_3

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [41]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_3

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [42]:
%env TABLE_NAME=reviews_mor_simple_3

env: TABLE_NAME=reviews_mor_simple_3


In [43]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_3

env: TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_simple_3


In [44]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 3

reviews_mor_simple_3


In [45]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [46]:
%%sql
drop table if exists ${TABLE_NAME}_ro

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [47]:
%%sql
drop table if exists ${TABLE_NAME}_rt

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [48]:
%%sql
create table if not exists ${TABLE_NAME} (
    review_id string, 
    star_rating long, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'mor',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175',
    -- hoodie.compact.inline = 'false',
    -- hoodie.compact.schedule.inline = 'false',
    hoodie.logfile.max.size = '512000',
    hoodie.compact.inline.max.delta.commits = '3'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 4.2. Batch 1 - Insert

In [49]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1999

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [50]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103916169 │ deltacommit │ COMPLETED │ 03-09 10:39 │ 03-09 10:39 │ 03-09 10:39 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪════════

### 4.3. Batch 2 - Update

In [51]:
%%sql
-- 更新1999-01-01的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-01'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [52]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103916169 │ deltacommit │ COMPLETED │ 03-09 10:39 │ 03-09 10:39 │ 03-09 10:39 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309104024105 │ deltacommit │ COMPLETED │ 03-09 10:40 │ 03-09 10:40 │ 03-09 10:40 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        

### 4.4. Batch 3 - Update

In [53]:
%%sql
-- 更新1999-01-02的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    star_rating = 5
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [54]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103916169 │ deltacommit │ COMPLETED │ 03-09 10:39 │ 03-09 10:39 │ 03-09 10:39 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309104024105 │ deltacommit │ COMPLETED │ 03-09 10:40 │ 03-09 10:40 │ 03-09 10:40 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309104104569 │ deltacommit │ COMPLETED │ 03-09 10:41 │ 03-09 10:41 │ 03-09 10:41 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔══════════

### 4.5. Async Schedule Compaction

In [55]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class 'org.apache.hudi.utilities.HoodieCompactor' \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'schedule' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" \
  --hoodie-conf "hoodie.compact.inline.max.delta.commits=3" > ~/${TABLE_NAME}.compaction.schedule.out &>/dev/null

In [56]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103916169 │ deltacommit │ COMPLETED │ 03-09 10:39 │ 03-09 10:39 │ 03-09 10:39 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309104024105 │ deltacommit │ COMPLETED │ 03-09 10:40 │ 03-09 10:40 │ 03-09 10:40 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309104104569 │ deltacommit │ COMPLETED │ 03-09 10:41 │ 03-09 10:41 │ 03-09 10:41 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309104218455

### 4.6. Async Execute Compaction

In [57]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class "org.apache.hudi.utilities.HoodieCompactor" \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'execute' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" > ~/${TABLE_NAME}.compaction.execute.out &>/dev/null

In [58]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309103916169 │ deltacommit │ COMPLETED │ 03-09 10:39 │ 03-09 10:39 │ 03-09 10:39 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309104024105 │ deltacommit │ COMPLETED │ 03-09 10:40 │ 03-09 10:40 │ 03-09 10:40 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309104104569 │ deltacommit │ COMPLETED │ 03-09 10:41 │ 03-09 10:41 │ 03-09 10:41 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309104218455